# Week 2 Day 4 — CrewAI Multi-Agent Crew

Using CrewAI with the local catalog (`data/products.csv`) for a small budget recommendation brief.


## Task 1 — Multi-agent design

**Business task:** Research Wireless Mouse vs Mechanical Keyboard in the catalog, compare prices, and write a short stakeholder brief recommending the cheaper in-stock option.

| Agent | Role | Goal | Backstory (short) |
|-------|------|------|-------------------|
| Catalog Researcher | Find catalog facts | Accurate prices / stock from CSV | Catalog-only worker; no marketing copy |
| Pricing Analyst | Compare numbers | Clear cheaper option + gap | Uses calculator; does not write the final brief |
| Stakeholder Brief Writer | Writing | Clear manager-facing brief | Uses prior outputs only; no tools |

**Why multiple agents?** Research, math, and writing need different tools. Separate roles kept each step simpler.

**When one agent is enough:** one product lookup and a short answer.


In [1]:
from crewai_crew import (
    DEFAULT_REQUEST,
    build_sequential_crew,
    build_hierarchical_crew,
    run_crew,
    score_output,
    MODEL,
)

print("request:")
print(DEFAULT_REQUEST)
print("model:", f"openai/{MODEL} @ groq")


request:
A budget-conscious client needs a recommendation between Wireless Mouse and Mechanical Keyboard. Research the catalog, compare prices, and write a short stakeholder brief that recommends the cheaper in-stock option.
model: openai/openai/gpt-oss-20b @ groq


## Task 2 — Agents & tools

- **Researcher:** `search_catalog`, `list_in_stock_by_category` (catalog only)
- **Analyst:** `calculator` (math only)
- **Writer:** no tools (cannot invent new catalog numbers)

Tools match each role.


In [2]:
from crewai_crew import build_agents, get_llm

llm = get_llm()
researcher, analyst, writer = build_agents(llm)
for a in (researcher, analyst, writer):
    tools = [t.name for t in (a.tools or [])]
    print(f"{a.role} -> tools={tools}")


C:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\fastapi\applications.py:18: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  from fastapi.exception_handlers import (
C:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\fastapi\applications.py:30: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  from fastapi.openapi.utils import get_openapi


Catalog Researcher -> tools=['search_catalog', 'list_in_stock_by_category']
Pricing Analyst -> tools=['calculator']
Stakeholder Brief Writer -> tools=[]


## Task 3 — Sequential process

Tasks are chained with `context` so later agents see earlier outputs.

Researcher output was free-form at first, so the analyst sometimes missed prices.
I added `CATALOG_FACTS` / `PRICE_ANALYSIS` JSON in `expected_output` to make the numbers easy to pass along.


In [3]:
seq = run_crew("sequential")
print("elapsed_sec:", seq["elapsed_sec"])
print("usage:", seq["usage"])
print("approx_cost_usd:", seq["approx_cost_usd"])
print("\n--- sequential output ---\n")
print(seq["output"])
print("\n--- scores ---")
print(score_output(seq["output"]))


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 07784780-6976-433e-a842-6ace5ab9c909                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Business request:                                                                                        │
│  A budget-conscious client needs a recommendation between Wireless Mouse and Mechanical Keyboard. Research the  │
│  catalog, compare prices, and write a short stakeholder brief that recommends the cheaper in-stock option.      │
│                                                                                                                 │
│  Use search_catalog (and list_in_stock_by_category if useful) to gather facts for Wireless Mouse and            │
│  Mechanical Keyboard. Do not invent prices.                                                                     │
│  ID: d5ffc728-9857-4b6f-bbbb-a63bef3fba5f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Catalog Researcher                                                                                      │
│                                                                                                                 │
│  Task: Business request:                                                                                        │
│  A budget-conscious client needs a recommendation between Wireless Mouse and Mechanical Keyboard. Research the  │
│  catalog, compare prices, and write a short stakeholder brief that recommends the cheaper in-stock option.      │
│                                                                                                                 │
│  Use search_catalog (and list_in_stock_by_category if useful) to gather facts for Wireless Mouse and            │
│  Mechanical Keyboard. Do not invent prices.                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_catalog                                                                                           │
│  Args: {'query': 'Wireless Mouse'}                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_catalog executed with result: {
  "query": "Wireless Mouse",
  "matches": [
    {
      "product_id": "P001",
      "name": "Wireless Mouse",
      "category": "accessories",
      "price_usd": "18.99",
      "in_stock": "true"
  ...


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stakeholder Brief Writer                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Recommendation                                                                                              │
│  Opt for the Wireless Mouse (P001). It delivers the essential functionality required while keeping costs low.   │
│                                                                                                                 │
│  ## Price comparison                                                                                            │
│  The Wireless Mouse is priced at $18.99, whereas the Mechanical Keyboard (P002) costs $79.50. The price gap is  │
│  $60.51, making the mouse the far more economical choice.                                                       │
│                                                                                                                 │
│  ## Stock                                                                                                       │
│  Both products are currently in stock, ensuring immediate availability for the client.                          │
│                                                                                                                 │
│  ## Why this fits budget                                                                                        │
│  Choosing the Wireless Mouse provides the necessary input device at a fraction of the cost, aligning perfectly  │
│  with a budget‑conscious purchasing strategy.                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using CATALOG_FACTS and PRICE_ANALYSIS from earlier tasks, write a stakeholder-ready brief. Include:     │
│  recommendation, price gap, stock note, and one sentence why this fits a budget-conscious client.               │
│  Agent: Stakeholder Brief Writer                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

elapsed_sec: 4.74
usage: {'total_tokens': 11787, 'prompt_tokens': 8916, 'completion_tokens': 2871, 'raw': 'total_tokens=11787 prompt_tokens=8916 cached_prompt_tokens=0 completion_tokens=2871 reasoning_tokens=1458 cache_creation_tokens=0 successful_requests=18', 'successful_requests': 18}
approx_cost_usd: 0.0007329000000000001

--- sequential output ---

## Recommendation  
Opt for the Wireless Mouse (P001). It delivers the essential functionality required while keeping costs low.

## Price comparison  
The Wireless Mouse is priced at $18.99, whereas the Mechanical Keyboard (P002) costs $79.50. The price gap is $60.51, making the mouse the far more economical choice.

## Stock  
Both products are currently in stock, ensuring immediate availability for the client.

## Why this fits budget  
Choosing the Wireless Mouse provides the necessary input device at a fraction of the cost, aligning perfectly with a budget‑conscious purchasing strategy.

--- scores ---
{'factual_grounding': 1, 'com

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 07784780-6976-433e-a842-6ace5ab9c909                                                                       │
│  Final Output: ## Recommendation                                                                                │
│  Opt for the Wireless Mouse (P001). It delivers the essential functionality required while keeping costs low.   │
│                                                                                                                 │
│  ## Price comparison                                                                                            │
│  The Wireless Mouse is priced at $18.99, whereas the Mechanical Keyboard (P002) costs $79.50. The price gap is  │
│  $60.51, making the mouse the far more economical choice.                                                       │
│                                                                                                                 │
│  ## Stock                                                                                                       │
│  Both products are currently in stock, ensuring immediate availability for the client.                          │
│                                                                                                                 │
│  ## Why this fits budget                                                                                        │
│  Choosing the Wireless Mouse provides the necessary input device at a fraction of the cost, aligning perfectly  │
│  with a budget‑conscious purchasing strategy.                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Task 4 — Hierarchical process

Same agents/tasks, but `Process.hierarchical` with a Project Manager that delegates and reviews.


In [4]:
hier = run_crew("hierarchical")
print("elapsed_sec:", hier["elapsed_sec"])
print("usage:", hier["usage"])
print("approx_cost_usd:", hier["approx_cost_usd"])
print("\n--- hierarchical output ---\n")
print(hier["output"])
print("\n--- scores ---")
print(score_output(hier["output"]))


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6317888e-b779-42aa-9324-88bb83b8ba76                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Business request:                                                                                        │
│  A budget-conscious client needs a recommendation between Wireless Mouse and Mechanical Keyboard. Research the  │
│  catalog, compare prices, and write a short stakeholder brief that recommends the cheaper in-stock option.      │
│                                                                                                                 │
│  Use search_catalog (and list_in_stock_by_category if useful) to gather facts for Wireless Mouse and            │
│  Mechanical Keyboard. Do not invent prices.                                                                     │
│  ID: a48ea29b-fb55-4b5e-ba34-702bc25e7e0e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Project Manager                                                                                         │
│                                                                                                                 │
│  Task: Business request:                                                                                        │
│  A budget-conscious client needs a recommendation between Wireless Mouse and Mechanical Keyboard. Research the  │
│  catalog, compare prices, and write a short stakeholder brief that recommends the cheaper in-stock option.      │
│                                                                                                                 │
│  Use search_catalog (and list_in_stock_by_category if useful) to gather facts for Wireless Mouse and            │
│  Mechanical Keyboard. Do not invent prices.                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#3) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_catalog                                                                                           │
│  Args: {'query': 'Wireless Mouse'}                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_catalog executed with result: {
  "query": "Wireless Mouse",
  "matches": [
    {
      "product_id": "P001",
      "name": "Wireless Mouse",
      "category": "accessories",
      "price_usd": "18.99",
      "in_stock": "true"
  ...


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Project Manager                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Recommendation                                                                                              │
│  We recommend the Wireless Mouse (P001) as the optimal choice for the client’s budget constraints.              │
│                                                                                                                 │
│  ## Price comparison                                                                                            │
│  - Wireless Mouse: $18.99                                                                                       │
│  - Mechanical Keyboard: $79.50                                                                                  │
│  The price difference is $60.51, making the mouse significantly more affordable.                                │
│                                                                                                                 │
│  ## Stock                                                                                                       │
│  Both items are currently in stock, ensuring immediate delivery.                                                │
│                                                                                                                 │
│  ## Why this fits budget                                                                                        │
│  The Wireless Mouse delivers essential functionality—wireless connectivity, ergonomic design, and reliable      │
│  performance—at a fraction of the cost of the Mechanical Keyboard. Its $18.99 price point allows the client to  │
│  allocate remaining funds to other critical needs while still providing a high‑quality input device.            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using CATALOG_FACTS and PRICE_ANALYSIS from earlier tasks, write a stakeholder-ready brief. Include:     │
│  recommendation, price gap, stock note, and one sentence why this fits a budget-conscious client.               │
│  Agent: Project Manager                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

elapsed_sec: 51.18
usage: {'total_tokens': 45188, 'prompt_tokens': 30208, 'completion_tokens': 14980, 'raw': 'total_tokens=45188 prompt_tokens=30208 cached_prompt_tokens=0 completion_tokens=14980 reasoning_tokens=10492 cache_creation_tokens=0 successful_requests=40', 'successful_requests': 40}
approx_cost_usd: 0.0030084000000000005

--- hierarchical output ---

## Recommendation  
We recommend the Wireless Mouse (P001) as the optimal choice for the client’s budget constraints.

## Price comparison  
- Wireless Mouse: $18.99  
- Mechanical Keyboard: $79.50  
The price difference is $60.51, making the mouse significantly more affordable.

## Stock  
Both items are currently in stock, ensuring immediate delivery.

## Why this fits budget  
The Wireless Mouse delivers essential functionality—wireless connectivity, ergonomic design, and reliable performance—at a fraction of the cost of the Mechanical Keyboard. Its $18.99 price point allows the client to allocate remaining funds to other cri

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6317888e-b779-42aa-9324-88bb83b8ba76                                                                       │
│  Final Output: ## Recommendation                                                                                │
│  We recommend the Wireless Mouse (P001) as the optimal choice for the client’s budget constraints.              │
│                                                                                                                 │
│  ## Price comparison                                                                                            │
│  - Wireless Mouse: $18.99                                                                                       │
│  - Mechanical Keyboard: $79.50                                                                                  │
│  The price difference is $60.51, making the mouse significantly more affordable.                                │
│                                                                                                                 │
│  ## Stock                                                                                                       │
│  Both items are currently in stock, ensuring immediate delivery.                                                │
│                                                                                                                 │
│  ## Why this fits budget                                                                                        │
│  The Wireless Mouse delivers essential functionality—wireless connectivity, ergonomic design, and reliable      │
│  performance—at a fraction of the cost of the Mechanical Keyboard. Its $18.99 price point allows the client to  │
│  allocate remaining funds to other critical needs while still providing a high‑quality input device.            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Sequential vs hierarchical

| | Sequential | Hierarchical |
|--|------------|--------------|
| Pros | Simple, fewer tokens | Manager can catch weak steps |
| Cons | Brittle if early output is messy | Slower / costlier |
| Use when | Steps are linear and clear | You need review / re-assignment |


## Task 5 — Evaluation & cost

Success criteria (0/1 each):
1. **factual_grounding** — real catalog prices appear (18.99 / 79.50)
2. **completeness** — recommendation + stock + price signal
3. **tone_structure** — clear brief structure, concise

Run the crew a couple more times and score each output. Compare cost/latency to Day 3 LangGraph (single graph, usually cheaper for this same catalog compare).


In [5]:
# Score the two runs above + one extra sequential run
runs = [
    ("sequential_1", seq),
    ("hierarchical_1", hier),
]
extra = run_crew("sequential")
runs.append(("sequential_2", extra))

rows = []
for name, r in runs:
    s = score_output(r["output"])
    rows.append({
        "run": name,
        "elapsed_sec": r["elapsed_sec"],
        "prompt_tokens": r["usage"].get("prompt_tokens"),
        "completion_tokens": r["usage"].get("completion_tokens"),
        "total_tokens": r["usage"].get("total_tokens"),
        "approx_cost_usd": r["approx_cost_usd"],
        **s,
    })

import json
print(json.dumps(rows, indent=2))
print("\nCompared with Day 3 LangGraph: one graph run is usually cheaper/faster for this same compare.")


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: afe5261c-30c5-4366-a444-426c80528929                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Business request:                                                                                        │
│  A budget-conscious client needs a recommendation between Wireless Mouse and Mechanical Keyboard. Research the  │
│  catalog, compare prices, and write a short stakeholder brief that recommends the cheaper in-stock option.      │
│                                                                                                                 │
│  Use search_catalog (and list_in_stock_by_category if useful) to gather facts for Wireless Mouse and            │
│  Mechanical Keyboard. Do not invent prices.                                                                     │
│  ID: e8989cb6-9d71-4e28-ac56-1d245c1cfb93                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Catalog Researcher                                                                                      │
│                                                                                                                 │
│  Task: Business request:                                                                                        │
│  A budget-conscious client needs a recommendation between Wireless Mouse and Mechanical Keyboard. Research the  │
│  catalog, compare prices, and write a short stakeholder brief that recommends the cheaper in-stock option.      │
│                                                                                                                 │
│  Use search_catalog (and list_in_stock_by_category if useful) to gather facts for Wireless Mouse and            │
│  Mechanical Keyboard. Do not invent prices.                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:OpenAI API call failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01jkz9qq6jek6aa27sqxcq7cp1` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6582, Requested 1446. Please try again in 210ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


ERROR:root:OpenAI API call failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01jkz9qq6jek6aa27sqxcq7cp1` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6582, Requested 1446. Please try again in 210ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stakeholder Brief Writer                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Recommendation                                                                                              │
│  Purchase the **Wireless Mouse (P001)**. It offers the same functionality as the Mechanical Keyboard at a       │
│  fraction of the cost and is immediately available.                                                             │
│                                                                                                                 │
│  ## Price comparison                                                                                            │
│  The Mechanical Keyboard is **$60.51** more expensive than the Wireless Mouse, making the mouse the clear       │
│  cost‑saving choice.                                                                                            │
│                                                                                                                 │
│  ## Stock                                                                                                       │
│  Both products are currently in stock, ensuring quick delivery for the client.                                  │
│                                                                                                                 │
│  ## Why this fits budget                                                                                        │
│  The Wireless Mouse delivers essential performance at a low price, aligning perfectly with a budget‑conscious   │
│  client’s needs.                                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using CATALOG_FACTS and PRICE_ANALYSIS from earlier tasks, write a stakeholder-ready brief. Include:     │
│  recommendation, price gap, stock note, and one sentence why this fits a budget-conscious client.               │
│  Agent: Stakeholder Brief Writer                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[
  {
    "run": "sequential_1",
    "elapsed_sec": 4.74,
    "prompt_tokens": 8916,
    "completion_tokens": 2871,
    "total_tokens": 11787,
    "approx_cost_usd": 0.0007329000000000001,
    "factual_grounding": 1,
    "completeness": 1,
    "tone_structure": 1,
    "total": 3
  },
  {
    "run": "hierarchical_1",
    "elapsed_sec": 51.18,
    "prompt_tokens": 30208,
    "completion_tokens": 14980,
    "total_tokens": 45188,
    "approx_cost_usd": 0.0030084000000000005,
    "factual_grounding": 1,
    "completeness": 1,
    "tone_structure": 1,
    "total": 3
  },
  {
    "run": "sequential_2",
    "elapsed_sec": 32.99,
    "prompt_tokens": 9144,
    "completion_tokens": 2694,
    "total_tokens": 11838,
    "approx_cost_usd": 0.0007266,
    "factual_grounding": 0,
    "completeness": 1,
    "tone_structure": 1,
    "total": 2
  }
]

Compared with Day 3 LangGraph: one graph run is usually cheaper/faster for this same compare.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: afe5261c-30c5-4366-a444-426c80528929                                                                       │
│  Final Output: ## Recommendation                                                                                │
│  Purchase the **Wireless Mouse (P001)**. It offers the same functionality as the Mechanical Keyboard at a       │
│  fraction of the cost and is immediately available.                                                             │
│                                                                                                                 │
│  ## Price comparison                                                                                            │
│  The Mechanical Keyboard is **$60.51** more expensive than the Wireless Mouse, making the mouse the clear       │
│  cost‑saving choice.                                                                                            │
│                                                                                                                 │
│  ## Stock                                                                                                       │
│  Both products are currently in stock, ensuring quick delivery for the client.                                  │
│                                                                                                                 │
│  ## Why this fits budget                                                                                        │
│  The Wireless Mouse delivers essential performance at a low price, aligning perfectly with a budget‑conscious   │
│  client’s needs.                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Cost and complexity

Sequential keeps research, math, and writing separate so I can check each step. Hierarchical used more time/tokens and only helps if the manager actually catches mistakes. Day 3 LangGraph is better for loops/interrupts; CrewAI is better for role handoffs. For this brief, sequential is enough.
